# 🧠 LOGOS — Vedic-Physics Hybrid LLM
### Complete Training on Kaggle GPU

**Kuch bhi bahar se download nahi karna — sab Kaggle ke andar hai**

| Component | Source |
|-----------|--------|
| Code | GitHub clone (public repo) |
| Dataset | Kaggle Dataset (TinyStories — already available) |
| GPU | Kaggle T4 (free 30hr/week) |
| Output | Kaggle /kaggle/working/ folder |

---
## ✅ STEP 1 — Environment Check

In [ ]:
# GPU, compiler, cmake sab check karo
import os, subprocess, glob

print("=== GPU ===")
os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

print("\n=== Compiler ===")
os.system("g++ --version | head -1")

print("\n=== CMake ===")
os.system("cmake --version | head -1")

print("\n=== CPU Cores ===")
os.system("nproc")

print("\n=== Disk Space ===")
os.system("df -h /kaggle/working")

---
## ✅ STEP 2 — Dataset Setup

### Kaggle pe dataset add karne ka tarika:
1. Right panel mein **"+ Add Data"** button click karo
2. Search karo: **`tinystories`**
3. **"roneneldan/tinystories"** select karo → Add
4. Ya neeche wala cell chalao — wo automatically Kaggle datasets check karega

**Agar dataset nahi milta** to cell 2b chalao — synthetic data generate hoga (training ke liye enough hai demo ke liye)

In [ ]:
# Cell 2a — Dataset dhundo (Kaggle input folders)
import os

DATASET_PATH = None

# Kaggle pe common dataset locations
search_paths = [
    "/kaggle/input/tinystories",
    "/kaggle/input/tiny-stories",
    "/kaggle/input/roneneldan-tinystories",
    "/kaggle/input",
]

# Saare .txt files dhundo
found_files = []
for path in search_paths:
    if os.path.exists(path):
        print(f"Found folder: {path}")
        for root, dirs, files in os.walk(path):
            for f in files:
                full = os.path.join(root, f)
                size = os.path.getsize(full)
                print(f"  {full}  ({size//1024//1024} MB)")
                if f.endswith('.txt') and size > 1000:
                    found_files.append((full, size))

if found_files:
    # Sabse bada txt file use karo
    found_files.sort(key=lambda x: x[1], reverse=True)
    DATASET_PATH = found_files[0][0]
    print(f"\n✅ Dataset found: {DATASET_PATH}")
    print(f"   Size: {found_files[0][1]//1024//1024} MB")
else:
    print("⚠️  Kaggle dataset nahi mila — Cell 2b chalao")

In [ ]:
# Cell 2b — Agar Kaggle dataset nahi mila to synthetic data banao
# (Skip karo agar DATASET_PATH upar set ho gaya)

if DATASET_PATH is None:
    print("Synthetic training data generate kar raha hai...")
    
    stories = [
        "Once upon a time there was a little girl named Lily who loved to play in the garden. She had a dog named Max who was always by her side. One day they found a magical flower that could talk. The flower said hello and Lily was surprised. Max barked happily at the flower. They became best friends and visited every day.",
        "Tom was a small boy who liked to read books about dinosaurs. His favorite was the Tyrannosaurus Rex. He wanted to be a scientist when he grew up. Every day after school he would go to the library and read more books. His teacher was very proud of him.",
        "The little rabbit lived in a cozy burrow under an old oak tree. Every morning the rabbit would hop to the stream to drink fresh water. One day the rabbit met a wise old owl who taught him about the stars in the sky. The rabbit listened carefully and learned many things.",
        "Sara loved to paint pictures of the ocean. She would sit by the window and watch the waves crash on the shore. Her mother would bring her colored pencils and paper. Sara dreamed of one day sailing on a big ship to see the world.",
        "The friendly dragon lived on top of a green mountain. All the animals in the forest were afraid of him at first. But when they got to know him they discovered he was very kind. He would use his fire breath to warm their homes in winter.",
        "Ben and his sister Emma liked to build things with wooden blocks. They built tall towers and bridges and castles. Their father helped them make a whole city on the living room floor. When it was time for bed they carefully put each block away.",
        "The old lighthouse stood at the edge of the sea. Its light guided ships safely through the dark night. A young keeper named Jack lived there alone. Every evening he climbed the long spiral stairs to light the great lamp.",
        "Maya had a little garden of her own. She planted tomatoes and sunflowers and mint. Every morning she watered her plants carefully. When summer came the sunflowers grew very tall and the tomatoes turned bright red.",
    ]
    
    synthetic_path = "/kaggle/working/synthetic_dataset.txt"
    with open(synthetic_path, 'w') as f:
        # 10000 baar repeat karo — enough training data
        import random
        random.seed(42)
        for _ in range(10000):
            story = random.choice(stories)
            f.write(story + "\n\n")
    
    size_mb = os.path.getsize(synthetic_path) / 1024 / 1024
    DATASET_PATH = synthetic_path
    print(f"✅ Synthetic dataset ready: {size_mb:.1f} MB")
    print(f"   Path: {DATASET_PATH}")

print(f"\n🎯 Final DATASET_PATH = {DATASET_PATH}")

In [ ]:
# Dataset working directory mein copy/link karo
WORK_DIR = "/kaggle/working/LOGOS"

# Pehle 50MB use karo (CPU training ke liye manageable)
TRAIN_FILE = "/kaggle/working/dataset.txt"

os.system(f"head -c 52428800 '{DATASET_PATH}' > {TRAIN_FILE}")
size = os.path.getsize(TRAIN_FILE)
print(f"✅ Training file ready: {size//1024//1024} MB")
print(f"   Words: ", end="")
os.system(f"wc -w {TRAIN_FILE}")

---
## ✅ STEP 3 — Clone & Build LOGOS

In [ ]:
import os

WORK_DIR = "/kaggle/working/LOGOS"

# Clone repo
if not os.path.exists(WORK_DIR):
    print("Cloning LOGOS...")
    ret = os.system("git clone https://github.com/Vikas8719/LOGOS.git " + WORK_DIR)
    if ret != 0:
        print("❌ Clone failed — check repo URL")
    else:
        print("✅ Clone successful")
else:
    print("Repo already exists — pulling latest...")
    os.system(f"cd {WORK_DIR} && git pull")

os.chdir(WORK_DIR)
print(f"\nWorking dir: {os.getcwd()}")
os.system("ls include/ src/")

In [ ]:
os.chdir(WORK_DIR)

print("Building LOGOS...")
ret = os.system("""
    cmake -B build \
        -DCMAKE_BUILD_TYPE=Release \
        -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \
        -DCMAKE_CXX_STANDARD=20 \
    && cmake --build build --parallel $(nproc)
""")

if ret == 0:
    print("\n✅ Build SUCCESS")
    os.system("ls -lh build/logos")
else:
    print("\n❌ Build FAILED — error log dekho")

---
## ✅ STEP 4 — Unit Tests

In [ ]:
os.chdir(WORK_DIR)
print("Running all tests...")
os.system("./build/logos --test")
os.system("./build/logos --forward")
os.system("./build/logos --benchmark")

---
## 🚀 STEP 5 — TRAIN LOGOS

**Expected loss curve:**
```
Step     0 → Loss: ~8.3  (random weights = log(vocab_size))
Step   500 → Loss: ~5.0  (patterns seekhna shuru)
Step  2000 → Loss: ~3.5  (words structure samajh raha)
Step 10000 → Loss: ~2.5  (meaningful text)
```

In [ ]:
import subprocess, sys, re, time
os.chdir(WORK_DIR)

# Dataset link
TRAIN_FILE = "/kaggle/working/dataset.txt"
if not os.path.exists(TRAIN_FILE):
    os.system(f"cp '{DATASET_PATH}' {TRAIN_FILE}")

# Training log file
LOG_FILE = "/kaggle/working/training_log.txt"

print("🚀 LOGOS Training START")
print(f"   Dataset: {TRAIN_FILE}")
print(f"   Log: {LOG_FILE}")
print("   Ctrl+C se rok sakte ho anytime\n")

steps_log = []
losses_log = []

process = subprocess.Popen(
    ["./build/logos", "--train", TRAIN_FILE],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    cwd=WORK_DIR
)

with open(LOG_FILE, 'w') as log:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            # Loss parse karo real-time
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                steps_log.append(int(m.group(1)))
                losses_log.append(float(m.group(2)))
    except KeyboardInterrupt:
        process.terminate()
        print("\n⏹️  Training stopped")

process.wait()
print(f"\n✅ Training done | {len(steps_log)} log points captured")
if losses_log:
    print(f"   Start loss: {losses_log[0]:.4f}")
    print(f"   Final loss: {losses_log[-1]:.4f}")
    print(f"   Improvement: {losses_log[0]-losses_log[-1]:.4f}")

---
## ✅ STEP 6 — Loss Curve Plot

In [ ]:
import matplotlib.pyplot as plt
import re

# Log file se parse karo
LOG_FILE = "/kaggle/working/training_log.txt"
steps_log, losses_log = [], []

if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        for line in f:
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                steps_log.append(int(m.group(1)))
                losses_log.append(float(m.group(2)))

if steps_log:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curve
    ax1.plot(steps_log, losses_log, 'b-', linewidth=1.5, alpha=0.8)
    ax1.set_title('LOGOS Training Loss\n(Langevin Physics Optimizer)', fontsize=13)
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Cross-Entropy Loss')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=losses_log[-1], color='r', linestyle='--',
                label=f'Final: {losses_log[-1]:.3f}')
    ax1.legend()

    # Perplexity curve
    import math
    perplexities = [math.exp(min(l, 10)) for l in losses_log]
    ax2.plot(steps_log, perplexities, 'g-', linewidth=1.5, alpha=0.8)
    ax2.set_title('Perplexity\n(Lower = Better)', fontsize=13)
    ax2.set_xlabel('Steps')
    ax2.set_ylabel('Perplexity')
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')

    plt.tight_layout()
    plt.savefig('/kaggle/working/loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Start  → Loss: {losses_log[0]:.4f} | Perplexity: {math.exp(losses_log[0]):.1f}")
    print(f"Final  → Loss: {losses_log[-1]:.4f} | Perplexity: {math.exp(min(losses_log[-1],10)):.1f}")
    print(f"Total steps: {steps_log[-1]}")
else:
    print("Training log nahi mila — pehle Step 5 chalao")

---
## ✅ STEP 7 — Evaluate & Generate Text

In [ ]:
os.chdir(WORK_DIR)

# Latest checkpoint
checkpoints = sorted(glob.glob("/kaggle/working/LOGOS/*.bin") +
                     glob.glob("/kaggle/working/*.bin"))
checkpoints = [c for c in checkpoints if 'vocab' not in c]

print("Checkpoints found:")
for c in checkpoints:
    print(f"  {c}  ({os.path.getsize(c)//1024//1024} MB)")

if checkpoints:
    latest = checkpoints[-1]
    print(f"\n📊 Evaluating: {latest}")
    os.system(f"./build/logos --eval {TRAIN_FILE} {latest}")

    print("\n" + "="*50)
    print("TEXT GENERATION")
    print("="*50)
    for prompt in ["Once upon a time", "The little girl", "Tom liked to"]:
        print(f"\nPrompt: '{prompt}'")
        os.system(f'./build/logos --generate {latest} "{prompt}"')
else:
    print("\n⚠️  Checkpoint nahi mila — pehle training chalao (Step 5)")

---
## ✅ STEP 8 — Save Output

In [ ]:
import shutil
os.chdir(WORK_DIR)

OUT = "/kaggle/working/logos_trained"
os.makedirs(OUT, exist_ok=True)

# Checkpoints
for f in glob.glob("*.bin"):
    shutil.copy(f, OUT)
    print(f"✅ {f}")

# Binary + vocab
shutil.copy("build/logos", OUT)
if os.path.exists("vocab.bin"): shutil.copy("vocab.bin", OUT)
if os.path.exists("/kaggle/working/loss_curve.png"):
    shutil.copy("/kaggle/working/loss_curve.png", OUT)
if os.path.exists("/kaggle/working/training_log.txt"):
    shutil.copy("/kaggle/working/training_log.txt", OUT)

print(f"\n=== Saved to {OUT} ===")
os.system(f"ls -lh {OUT}")
print("\n✅ Kaggle 'Output' tab mein download kar sakte ho!")